# Snippet from Math-Lyapunov-Stability.md


In [ ]:
from compitum.control import LyapunovController

# The REAL constructor is (kappa: float = 0.1, r0: float = 1.0,
# integral_gain: float = 0.005) -- there is no initial_radius/shrink_factor/
# expand_factor/ema_alpha/threshold parameterization, and there is NO
# lyapunov_gate() method that vetoes an update based on the sign of a raw
# delta-V. The real update() method takes a non-negative drift observation
# `d_star` plus a `grad_norm`, and always returns (eta_cap, status_dict) --
# it never accepts/rejects anything itself. It maintains an EMA of the
# drift (`drift_ema`) and an integral term (`drift_integral`), and adjusts
# `trust_radius` by a conditional *0.8/*1.1 multiplier plus a linear
# integral-gain subtraction (see control.py) -- not a delta-V accept/reject
# gate. The "gate" framing below is illustrative bookkeeping built on top
# of the real controller, not a literal transcription of control.py.

def validate_trajectory():
    """Walk the real LyapunovController through a noisy energy trajectory."""

    # Simulated free energy trajectory with noise
    E_trajectory = [
        1.00,  # Initial state
        0.75,  # Good descent
        0.60,  # Continued descent
        0.65,  # Small increase (noise violation)
        0.50,  # Recovery descent
        0.55,  # Another violation
        0.40,  # Strong descent
        0.35,  # Convergence
    ]

    controller = LyapunovController(kappa=0.1, r0=1.0, integral_gain=0.005)

    print("=" * 60)
    print("LyapunovController Trajectory Walkthrough")
    print("=" * 60)

    for t in range(1, len(E_trajectory)):
        E_t = E_trajectory[t - 1]
        E_tp1 = E_trajectory[t]
        delta_V = E_tp1 - E_t

        # d_star is fed as the non-negative drift magnitude (the real API
        # expects a distance-like, non-negative quantity); grad_norm is
        # held fixed here for illustration.
        d_star = max(delta_V, 0.0)
        grad_norm = 1.0
        eta_cap, status = controller.update(d_star, grad_norm)

        direction = "energy up" if delta_V > 0 else "energy down"
        print(f"\nStep {t}: {direction}")
        print(f"  E_t={E_t:.3f} -> E_t+1={E_tp1:.3f}, delta_V={delta_V:+.3f}")
        print(
            f"  eta_cap={eta_cap:.4f}, trust_radius={status['trust_radius']:.4f}, "
            f"drift_ema={status['drift_ema']:.4f}"
        )

    print("\n" + "=" * 60)
    print("Summary Statistics")
    print("=" * 60)
    print(f"Final trust_radius: {controller.trust_radius:.4f}")
    print(f"Final drift_ema: {controller.drift_ema:.4f}")
    print(f"Final drift_integral: {controller.drift_integral:.4f}")

if __name__ == "__main__":
    validate_trajectory()
